# Watershed Analysis from a DEM with Pure Python

*Geography + AI Teaching Notebooks · 2 of 3*

Geographic Information Systems hide a lot of hydrology behind a few menu
clicks. This notebook opens the black box: it derives a stream network
and several terrain metrics **from a digital elevation model (DEM) using
nothing but NumPy**, so students can see exactly what each step does.

Everything runs on a synthetic DEM, so the notebook is fully
reproducible. Swapping in a real DEM (e.g. an open national elevation
dataset) requires changing only the first code cell.

**What students practise here**
1. Reading and visualising a DEM.
2. The D8 flow-direction algorithm.
3. Flow accumulation and stream-network extraction.
4. The hypsometric curve as a descriptor of basin form.
5. The Topographic Wetness Index.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

rng = np.random.default_rng(7)
GRID = 200

# A synthetic DEM: a tilted plane plus smoothed random relief.
yy, xx = np.mgrid[0:GRID, 0:GRID]
tilt = 1800 - 0.6 * xx - 0.5 * yy   # highland in one corner
relief = gaussian_filter(rng.normal(0, 60, (GRID, GRID)), sigma=5)
dem = np.clip(tilt + relief, 0, None)
print(f'DEM elevation range: {dem.min():.0f}-{dem.max():.0f} m')


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(dem, cmap='terrain', origin='lower')
plt.colorbar(im, ax=ax, label='Elevation (m)')
ax.set_title('Synthetic digital elevation model')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 1. Flow direction (the D8 algorithm)

D8 assigns each cell the direction of its steepest-descent neighbour —
one of eight compass directions. It is the foundation of almost every
GIS hydrology tool.


In [ ]:
def d8_flow_direction(dem):
    di = np.array([-1, -1, -1,  0, 0,  1, 1, 1])
    dj = np.array([-1,  0,  1, -1, 1, -1, 0, 1])
    dist = np.sqrt(di**2 + dj**2)
    direction = np.zeros(dem.shape, dtype=np.int8)
    slope = np.zeros(dem.shape)
    for k in range(8):
        shifted = np.roll(np.roll(dem, -di[k], axis=0), -dj[k], axis=1)
        s = (dem - shifted) / dist[k]
        mask = s > slope
        slope[mask] = s[mask]
        direction[mask] = k + 1
    return direction, slope

direction, slope = d8_flow_direction(dem)
print('Flow direction grid computed.')


## 2. Flow accumulation

Route one unit of water through every cell repeatedly. Cells that many
others drain into accumulate large values — these are the channels.


In [ ]:
def flow_accumulation(direction, n_iter=40):
    di = np.array([0, -1, -1, -1,  0, 0,  1, 1, 1])
    dj = np.array([0, -1,  0,  1, -1, 1, -1, 0, 1])
    acc = np.ones(direction.shape)
    for _ in range(n_iter):
        new_acc = np.ones(direction.shape)
        for k in range(1, 9):
            shifted = np.roll(np.roll(acc, di[k], axis=0), dj[k], axis=1)
            inflow = (np.roll(np.roll(direction, di[k], axis=0), dj[k], axis=1) == k)
            new_acc += shifted * inflow
        acc = new_acc
    return acc

acc = flow_accumulation(direction)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(np.log1p(acc), cmap='Blues', origin='lower')
plt.colorbar(im, ax=ax, label='log(accumulated cells)')
ax.set_title('Flow accumulation (log scale)')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 3. Extracting the stream network

Cells whose accumulation exceeds a threshold are treated as streams.
The threshold is a modelling choice — students should experiment with it.


In [ ]:
threshold = np.percentile(acc, 96)
stream = acc > threshold

fig, ax = plt.subplots(figsize=(6, 5))
ax.imshow(dem, cmap='terrain', origin='lower', alpha=0.7)
ax.imshow(np.where(stream, 1, np.nan), cmap='Blues', origin='lower', alpha=0.9)
ax.set_title('Extracted stream network')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## 4. The hypsometric curve

The hypsometric curve plots elevation against the fraction of the basin
lying above it. Its shape is a classic indicator of how 'young' or
'mature' a landscape is (Strahler, 1952).


In [ ]:
elevations = np.sort(dem.flatten())
cum_area = np.arange(elevations.size, 0, -1) / elevations.size
norm_z = (elevations - elevations.min()) / (elevations.max() - elevations.min())

fig, ax = plt.subplots(figsize=(5, 4))
ax.plot(cum_area, norm_z, lw=2)
ax.set_xlabel('Cumulative area fraction')
ax.set_ylabel('Normalised elevation')
ax.set_title('Hypsometric curve')
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()


## 5. The Topographic Wetness Index

TWI = ln(a / tan β) combines upslope area with local slope to predict
where soils stay wet. It is a quick first guess for saturation-prone
ground and riparian zones.


In [ ]:
tan_beta = np.tan(np.arctan(slope) + 1e-6)
twi = np.log((acc + 1) / (tan_beta + 1e-6))

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(twi, cmap='YlGnBu', origin='lower')
plt.colorbar(im, ax=ax, label='TWI')
ax.set_title('Topographic Wetness Index')
ax.set_xticks([]); ax.set_yticks([])
plt.tight_layout(); plt.show()


## Wrap-up

Each step above replaces several GIS menu clicks with a few transparent
lines of NumPy. Replacing the synthetic DEM with a real one — from any
open national elevation dataset — turns this notebook into the starting
point for a genuine watershed project.

### Suggested extensions for students

1. Load a real DEM with `rioxarray` and re-run every cell.
2. Delineate the watershed draining to a chosen outlet cell.
3. Compare the hypsometric curve of two contrasting basins.
4. Overlay a satellite-derived water index on the extracted network.
